In [2]:
import os
import numpy as np
import tensorflow as tf
import flwr as fl
from pathlib import Path

from tensorflow.keras import layers, models

2026-02-01 18:38:02.660434: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769971083.171889      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769971083.307690      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769971084.468748      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769971084.468796      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769971084.468799      55 computation_placer.cc:177] computation placer alr

ModuleNotFoundError: No module named 'flwr'

In [ ]:
### Dataset
DATA_DIR = "/kaggle/input/plantvillage-dataset/color"

# Federated settings
NUM_CLIENTS = 3
NUM_ROUNDS = 3
LOCAL_EPOCHS = 2

# Training settings
BATCH_SIZE = 32
IMG_SIZE = (160,160)


In [ ]:
def load_data(data_dir):
    data_dir = Path(data_dir)
    class_names = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])

    image_paths = []
    labels = []
    
    for label, class_name in enumerate(class_names):
        for img_path in (data_dir / class_name).glob("*"):
            image_paths.append(str(img_path))
            labels.append(label)

    return image_paths, labels
        


In [ ]:
img_paths, img_labels = load_data(DATA_DIR)


NUM_SAMPLES = len(img_paths)
NUM_CLASSES = len(np.unique(img_labels))

print("Classes:", NUM_CLASSES)
print("Total images:", NUM_SAMPLES)


In [ ]:
def split_indices(num_samples, num_clients):
    indices = np.arange(num_samples)
    np.random.shuffle(indices)
    return np.array_split(indices, num_clients)


In [ ]:
client_indices = split_indices(NUM_SAMPLES, NUM_CLIENTS)


In [ ]:
def make_dataset(indices):
    paths = [img_paths[i] for i in indices]
    labs = [img_labels[i] for i in indices]

    ds = tf.data.Dataset.from_tensor_slices((paths, labs))

    def load_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = img / 255.0
        return img, label

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return ds


In [ ]:
def create_model():
    data_augmentation = tf.keras.Sequential(
        [
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(0.1),
            tf.keras.layers.RandomZoom(0.1),
        ],
        name="data_augmentation",
    )
    preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
    IMG_SHAPE = (160,160,3)
    base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SHAPE,
                                               include_top=False,
                                               weights='imagenet')
    base_model.trainable = False
    global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
    prediction_layer = tf.keras.layers.Dense(38, activation="softmax")
    inputs = tf.keras.Input(shape=(160,160,3))
    x = data_augmentation(inputs)
    x = preprocess_input(x)
    x = base_model(x, training=False)
    x = global_average_layer(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = prediction_layer(x)
    model = tf.keras.Model(inputs, outputs)

    base_learning_rate = 0.0001
    model.compile(
        optimizer= tf.keras.optimizers.Adam(learning_rate=base_learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [ ]:
class FlowerClient(fl.client.NumPyClient):
    def __init__(self, model, train_ds):
        self.model = model
        self.train_ds = train_ds

    def get_parameters(self, config=None):
        return self.model.get_weights()

    def fit(self, parameters, config=None):
        self.model.set_weights(parameters)
        self.model.fit(self.train_ds, epochs=LOCAL_EPOCHS, verbose=0)
        return self.model.get_weights(), len(self.train_ds), {}

    def evaluate(self, parameters, config=None):
        self.model.set_weights(parameters)
        loss, acc = self.model.evaluate(self.train_ds, verbose=0)
        return loss, len(self.train_ds), {"accuracy": acc}


In [ ]:
def weighted_average(metrics):
    accuracies = []
    examples = []

    for num_examples, m in metrics:
        accuracies.append(m["accuracy"] * num_examples)
        examples.append(num_examples)

    return {"accuracy": sum(accuracies) / sum(examples)}


In [ ]:
def client_fn(cid):
    cid = int(cid)
    model = create_model()
    train_ds = make_dataset(client_indices[cid])
    return FlowerClient(model, train_ds)


In [ ]:
strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,
    fraction_evaluate=1.0,
    min_fit_clients=NUM_CLIENTS,
    min_evaluate_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
    evaluate_metrics_aggregation_fn=weighted_average
)


In [ ]:
 history = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy,
)

In [ ]:
print(history.__dict__)